# 3. Step 3: Model Creation, Hyperparameter Search, and Model Evaluation #

## Goals: ##
* Create train test splits and implement a RandomForestClassifier or a GradientBoostingClassifier 
* Hyperparameter tuning via GridSearchCV or RandomizedSearchCV
* Upon finding optimal hyperparameters, we re-train model using these hyperparameters, generate predictions for this new model, and output the subsequent F1 score for this classifier. 

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


In [ ]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay

RANDOM_STATE = 42

# --------------------
# Step 2: Transform
# --------------------
INPUT_PATH = "../data/caishen_bank_transactions.csv"
CLEAN_PATH = "../data/cleaned_caishen_bank_transactions.csv"

df = pd.read_csv(INPUT_PATH)

print("Initial shape:", df.shape)
print("Missing values:\n", df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())

if df.duplicated().any():
    df = df.drop_duplicates().copy()

for col in ["nameOrig", "nameDest"]:
    if col in df.columns:
        df = df.drop(columns=[col])

if "type" in df.columns:
    df = pd.get_dummies(df, columns=["type"], drop_first=True)

if {"oldbalanceOrg", "newbalanceOrig", "amount"}.issubset(df.columns):
    df["orig_balance_change"] = df["oldbalanceOrg"] - df["newbalanceOrig"]
    df["amount_minus_orig_change"] = df["amount"] - df["orig_balance_change"]
    df["amount_equals_orig_change"] = (df["amount"] == df["orig_balance_change"]).astype(int)
    df["orig_zero_after"] = (df["newbalanceOrig"] == 0).astype(int)

if {"oldbalanceDest", "newbalanceDest", "amount"}.issubset(df.columns):
    df["dest_balance_change"] = df["newbalanceDest"] - df["oldbalanceDest"]
    df["amount_minus_dest_change"] = df["amount"] - df["dest_balance_change"]
    df["dest_zero_before"] = (df["oldbalanceDest"] == 0).astype(int)
    df["dest_zero_after"] = (df["newbalanceDest"] == 0).astype(int)

if {"oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest"}.issubset(df.columns):
    df["net_flow_org"] = df["oldbalanceOrg"] - df["newbalanceOrig"]
    df["net_flow_dest"] = df["newbalanceDest"] - df["oldbalanceDest"]

for col in ["amount", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest"]:
    if col in df.columns:
        df[f"log1p_{col}"] = np.log1p(df[col].clip(lower=0))

if "isFlaggedFraud" in df.columns:
    df = df.drop(columns=["isFlaggedFraud"])

print("Cleaned shape:", df.shape)
df.to_csv(CLEAN_PATH, index=False)

# --------------------
# Step 3: Model
# --------------------
MODEL_RESULTS_PATH = "step3_model_results.csv"

df = pd.read_csv(CLEAN_PATH)

y = df["isFraud"]
X = df.drop(columns=["isFraud"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

baseline_model = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight="balanced"
)
baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_test)

baseline_metrics = {
    "model": "baseline_random_forest",
    "f1": f1_score(y_test, baseline_pred),
    "precision": precision_score(y_test, baseline_pred, zero_division=0),
    "recall": recall_score(y_test, baseline_pred, zero_division=0)
}

rf_param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [None, 5, 10, 20, 30],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": ["sqrt", "log2", None],
    "class_weight": [None, "balanced", "balanced_subsample"]
}

rf_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=rf_param_dist,
    n_iter=20,
    scoring="f1",
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)
rf_search.fit(X_train, y_train)
rf_best = rf_search.best_estimator_
rf_pred = rf_best.predict(X_test)

rf_metrics = {
    "model": "tuned_random_forest",
    "f1": f1_score(y_test, rf_pred),
    "precision": precision_score(y_test, rf_pred, zero_division=0),
    "recall": recall_score(y_test, rf_pred, zero_division=0)
}

results = pd.DataFrame([baseline_metrics, rf_metrics])
print(results)
print("Best RF params:", rf_search.best_params_)
print(classification_report(y_test, rf_pred, zero_division=0))

cm = confusion_matrix(y_test, rf_pred)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Tuned Random Forest Confusion Matrix")
plt.tight_layout()
plt.show()

results.to_csv(MODEL_RESULTS_PATH, index=False)

# --------------------
# Step 4: Report helper
# --------------------
results = pd.read_csv(MODEL_RESULTS_PATH)
print(results)

baseline = results.loc[results["model"] == "baseline_random_forest"].iloc[0]
tuned = results.loc[results["model"] == "tuned_random_forest"].iloc[0]

print(f"Baseline F1: {baseline['f1']:.4f}")
print(f"Baseline Precision: {baseline['precision']:.4f}")
print(f"Baseline Recall: {baseline['recall']:.4f}")
print(f"Tuned F1: {tuned['f1']:.4f}")
print(f"Tuned Precision: {tuned['precision']:.4f}")
print(f"Tuned Recall: {tuned['recall']:.4f}")

# --------------------
# Steps 5-7: Final report drafting helper
# --------------------
report_answers = {
    "q1_eda": "Fraud was highly imbalanced and concentrated in certain transaction types and balance-change patterns.",
    "q2_transform": "I dropped identifier-like columns, encoded type, and engineered balance-based features.",
    "q3_model": "I used a baseline Random Forest and tuned it with RandomizedSearchCV.",
    "q4_results": "The tuned model should be evaluated with F1, precision, and recall, not accuracy alone.",
    "q5_next_steps": "Next steps would include trying Gradient Boosting, threshold tuning, and feature importance review."
}

for k, v in report_answers.items():
    print(f"{k}: {v}\n")

Initial shape: (6362620, 11)
Missing values:
 step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64
Duplicate rows: 0
Cleaned shape: (6362620, 26)
Fitting 3 folds for each of 20 candidates, totalling 60 fits
